[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ManzarIMalik/in2stem-placement/blob/main/Intro_to_computer_vision.ipynb)

# What is Computer Vision

Computer Vision (CV) is the field of AI that teaches computers to "see" - to take in images or video and make sense of what's in them.

**Face unlock:** your phone compares the camera image against your stored face.
**Self-driving cars:** spotting pedestrians, lanes, and other vehicles in real time.
**Medicine:** helping doctors spot tumors in scans.
**Agriculture:** a drone counting crops or spotting disease from the air.

At its core, CV is about turning **pixels into understanding**. We'll build that understanding gradually in this notebook - starting with what an image actually *is* to a computer, through classic image-processing tricks, all the way to the deep learning models powering the applications above - with interactive widgets and "try it yourself" exercises along the way so the ideas actually stick.

In [ ]:
# Colab already ships with OpenCV, NumPy, and Matplotlib - nothing to install for this section.
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

# Images as Data

## Images Are Just Numbers

A computer has no idea what a "cat" or a "car" looks like. All it sees is a big grid of numbers.

* A **grayscale** image is a 2D grid (height x width) where each cell is a number from 0 (black) to 255 (white).
* A **color** image is a 3D grid (height x width x 3) - one 2D grid per color channel: Red, Green, and Blue.

Let's download a sample photo and look at the actual numbers behind it.

In [ ]:
import urllib.request

# A small, well-known sample image used throughout the Ultralytics docs
url = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(url, "/content/bus.jpg")

# OpenCV loads images in BGR order (not RGB!) - a historical quirk of the library
img_bgr = cv2.imread("/content/bus.jpg")
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print("Image shape (height, width, channels):", img.shape)
print("Data type:", img.dtype)
print("Total pixels:", img.shape[0] * img.shape[1])
print("Total numbers stored:", img.size, f"({img.nbytes/1024:.0f} KB as raw bytes)")
print("Top-left corner pixel values (R, G, B):", img[0, 0])

plt.imshow(img)
plt.title("Our sample image")
plt.axis("off")
plt.show()

**Try it yourself:** the coordinates below sit in the middle of the bus's blue side panel. Print the RGB values and read them as three numbers: which of the three is largest, and does that match the colour you can see?

Then deliberately break it. Set `row = 200` and re-run. You are now sampling something else entirely - can you work out what, and why nothing warned you?

In [ ]:
row, col = 560, 450  # TODO: change these coordinates and re-run
print(f"Pixel at (row={row}, col={col}):", img[row, col])

**What you should have seen:** something close to `[7, 75, 158]` - very little red, some green, a lot of blue. That is the bus's blue panel, in the only terms the computer has: three numbers.

If you left the coordinates at `(200, 300)` you got something like `[171, 151, 116]` instead - a muddy brown. That is not the bus at all, it is the **building behind it**. Row 200 is near the top of a 1080-pixel-tall photo, well above the roof of the bus. This is worth doing once on purpose: nothing warns you when you sample the wrong thing. No error, no empty result - just three plausible numbers describing something you did not mean to look at.

That is the point of this section. A photograph *looks* like a bus and a street and some pedestrians to you. To the computer it is a 1080 x 810 x 3 grid of integers, and the only difference between the bus and the building is that some of the numbers are different. Everything else in this notebook is built on getting from those numbers back to "that is a bus".

### Looking at Individual Color Channels

Since a color image is really three grids stacked together, we can pull out and view just the Red, Green, or Blue channel on its own. Try the widget below.

In [ ]:
from ipywidgets import interact, Dropdown

def show_channel(channel):
    """Displays a single color channel of the sample image as grayscale."""
    idx = {"Red": 0, "Green": 1, "Blue": 2}[channel]
    channel_data = img[:, :, idx]

    plt.figure(figsize=(6, 6))
    plt.imshow(channel_data, cmap="gray")
    plt.title(f"{channel} channel")
    plt.axis("off")
    plt.show()

interact(show_channel, channel=Dropdown(options=["Red", "Green", "Blue"], description="Channel:"));

## Color Spaces: RGB vs. HSV

RGB is great for displays, but it's awkward for *reasoning* about color - "yellow" can be many different mixes of R, G, and B depending on brightness. The **HSV** color space (Hue, Saturation, Value) separates color into:

* **Hue** - the actual color (0-179 in OpenCV), like a position on a color wheel.
* **Saturation** - how vivid vs. washed-out the color is.
* **Value** - how bright vs. dark it is.

This makes it much easier to say "find everything that's yellow" regardless of lighting.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = ["Hue", "Saturation", "Value"]
for i, (ax, title) in enumerate(zip(axes, titles)):
    ax.imshow(hsv[:, :, i], cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

**Try it yourself - color masking:** the bus in our photo is **blue**. Use the sliders below to find an HSV range that isolates *just* the blue bus and turns everything else black. This exact technique (`cv2.inRange`) is used in real applications like sports-ball tracking and traffic-sign detection.

The Hue slider is the one that matters most - in OpenCV's 0-179 scale, blue lives somewhere around 100-130. Drag it well away from there and watch the bus disappear entirely, which tells you something useful: if a mask comes back empty, your assumption about the colour is usually what is wrong, not the code.

When you have the bus, look hard at what *else* survived.

In [ ]:
from ipywidgets import IntRangeSlider

def color_mask(hue_range, sat_range, val_range):
    """Masks the image, keeping only pixels inside the given HSV range."""
    lower = np.array([hue_range[0], sat_range[0], val_range[0]])
    upper = np.array([hue_range[1], sat_range[1], val_range[1]])
    mask = cv2.inRange(hsv, lower, upper)
    result = cv2.bitwise_and(img, img, mask=mask)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(mask, cmap="gray")
    axes[0].set_title("Mask")
    axes[0].axis("off")
    axes[1].imshow(result)
    axes[1].set_title("Masked result")
    axes[1].axis("off")
    plt.show()

interact(
    color_mask,
    hue_range=IntRangeSlider(min=0, max=179, value=[100, 130], description="Hue:"),
    sat_range=IntRangeSlider(min=0, max=255, value=[80, 255], description="Sat:"),
    val_range=IntRangeSlider(min=0, max=255, value=[80, 255], description="Val:"),
);

**What you should have found:** a hue range of roughly **100 to 130** isolates the bus. Anything much outside that and it vanishes.

Now look carefully at what else survived the mask, because it matters more than the bus did. **The pedestrians' jeans are still there.** So are patches of sky and a few shop signs.

The mask did exactly what you asked. You asked for *blue pixels*, and it gave you every blue pixel in the photo - and denim is blue. It has no concept of "bus". It never did.

This is the ceiling of classical computer vision, and it is worth feeling rather than being told. Colour masking is fast, transparent, needs no training data and no GPU, and for a controlled problem - a bright orange ball on green grass, a red traffic sign - it is genuinely the right tool and still used in production today. But the moment two different things share a colour, it has nothing left to offer. You cannot fix this with better sliders. There is no hue range that means "bus".

Getting from "blue pixels" to "bus" is the entire subject of the second half of this notebook.

## Grayscale and Thresholding

A lot of classic computer vision works on grayscale images because it's simpler - one number per pixel instead of three. We can also make a decision for every pixel: "is this pixel light or dark?" That simple idea is called **thresholding**, and it's the basis of things like document scanners that turn a photo into black-and-white text.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Grayscale")
axes[0].axis("off")

axes[1].hist(gray.ravel(), bins=256, range=(0, 255))
axes[1].set_title("Pixel intensity histogram")
axes[1].set_xlabel("Pixel value (0=black, 255=white)")
plt.show()

In [ ]:
from ipywidgets import IntSlider

def show_threshold(threshold):
    """Turns every pixel either fully black or fully white based on a threshold."""
    _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    plt.figure(figsize=(6, 6))
    plt.imshow(binary, cmap="gray")
    plt.title(f"Threshold = {threshold}")
    plt.axis("off")
    plt.show()

interact(show_threshold, threshold=IntSlider(min=0, max=255, step=5, value=127));

Picking a threshold by hand is fiddly. **Otsu's method** automatically picks the threshold that best separates an image into two groups of pixels - no guessing required.

In [ ]:
otsu_value, otsu_binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f"Otsu automatically chose threshold = {otsu_value:.0f}")

plt.figure(figsize=(6, 6))
plt.imshow(otsu_binary, cmap="gray")
plt.title("Otsu's automatic threshold")
plt.axis("off")
plt.show()

# Classical Image Processing

## Filters and Convolutions

Most classic image processing (and, as we'll see later, deep learning) is built on one core operation: the **convolution**. A small grid of numbers called a **kernel** slides across the image. At each position, it multiplies its numbers with the pixels underneath and adds up the result into one new pixel.

Different kernels produce different effects - blurring, sharpening, or finding edges - just by changing the numbers in that small grid.

In [ ]:
# A blur kernel: every output pixel becomes the average of its neighbors
blur_kernel = np.ones((5, 5), dtype=np.float32) / 25

# A sharpen kernel: boosts the center pixel relative to its neighbors
sharpen_kernel = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0],
])

blurred = cv2.filter2D(img, -1, blur_kernel)
sharpened = cv2.filter2D(img, -1, sharpen_kernel)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [img, blurred, sharpened], ["Original", "Blurred (custom kernel)", "Sharpened (custom kernel)"]):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.show()

**Try it yourself:** define your own 3x3 kernel below (the numbers should usually add up to 1 so brightness stays the same) and see what effect it has. A couple of ideas: an "emboss" kernel `[[-2,-1,0],[-1,1,1],[0,1,2]]`, or an even stronger blur.

In [ ]:
my_kernel = np.array([
    [-2, -1, 0],
    [-1,  1, 1],
    [ 0,  1, 2],
])  # TODO: try your own numbers here

custom_result = cv2.filter2D(img, -1, my_kernel)
plt.figure(figsize=(6, 6))
plt.imshow(custom_result)
plt.title("Your custom kernel")
plt.axis("off")
plt.show()

OpenCV also ships ready-made filters so you don't have to hand-write the kernel every time - `GaussianBlur` (smooth blur) and `medianBlur` (great for removing "salt and pepper" noise) are two of the most common. Try changing the kernel size below and see how the blur gets stronger.

In [ ]:
def show_blur(kernel_size):
    """Applies a Gaussian blur with an adjustable kernel size."""
    k = kernel_size if kernel_size % 2 == 1 else kernel_size + 1  # kernel size must be odd
    blurred = cv2.GaussianBlur(img, (k, k), 0)
    plt.figure(figsize=(6, 6))
    plt.imshow(blurred)
    plt.title(f"Gaussian blur, kernel size = {k}")
    plt.axis("off")
    plt.show()

interact(show_blur, kernel_size=IntSlider(min=1, max=31, step=2, value=5));

## Edge Detection

Edges are places where pixel brightness changes sharply - the boundary of an object, for example. The **Sobel** operator estimates how quickly brightness changes horizontally (`dx`) and vertically (`dy`); combining the two gives us the overall edge strength.

In [ ]:
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobel_x, sobel_y)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [sobel_x, sobel_y, sobel_combined], ["Horizontal gradient (dx)", "Vertical gradient (dy)", "Combined edge strength"]):
    ax.imshow(np.abs(im), cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

The **Canny** edge detector (built on the same gradient idea, plus some extra cleanup steps) is a more reliable, ready-to-use alternative. It uses two thresholds: pixels above the high threshold are definitely edges, pixels below the low threshold are definitely not, and anything in between is only kept if it connects to a strong edge.

In [ ]:
def show_edges(low_threshold, high_threshold):
    """Runs Canny edge detection with adjustable thresholds."""
    edges = cv2.Canny(gray, low_threshold, high_threshold)
    plt.figure(figsize=(6, 6))
    plt.imshow(edges, cmap="gray")
    plt.title(f"Canny edges (low={low_threshold}, high={high_threshold})")
    plt.axis("off")
    plt.show()

interact(
    show_edges,
    low_threshold=IntSlider(min=0, max=255, step=5, value=50, description="Low:"),
    high_threshold=IntSlider(min=0, max=255, step=5, value=150, description="High:"),
);

## Morphological Operations

Thresholding and edge detection often leave behind small holes or stray noise pixels. **Morphological operations** clean these up by growing or shrinking white regions in a binary image:

* **Erosion** shrinks white regions (removes small white noise).
* **Dilation** grows white regions (fills small black holes).
* **Opening** = erosion then dilation (removes noise while keeping object size roughly the same).
* **Closing** = dilation then erosion (fills small holes while keeping object size roughly the same).

In [ ]:
from ipywidgets import RadioButtons

kernel_5x5 = np.ones((5, 5), np.uint8)

def show_morphology(operation, iterations):
    """Applies a morphological operation to the Otsu-thresholded bus image."""
    ops = {
        "Erosion": lambda im: cv2.erode(im, kernel_5x5, iterations=iterations),
        "Dilation": lambda im: cv2.dilate(im, kernel_5x5, iterations=iterations),
        "Opening": lambda im: cv2.morphologyEx(im, cv2.MORPH_OPEN, kernel_5x5, iterations=iterations),
        "Closing": lambda im: cv2.morphologyEx(im, cv2.MORPH_CLOSE, kernel_5x5, iterations=iterations),
    }
    result = ops[operation](otsu_binary)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(otsu_binary, cmap="gray")
    axes[0].set_title("Before")
    axes[0].axis("off")
    axes[1].imshow(result, cmap="gray")
    axes[1].set_title(f"After {operation.lower()} (x{iterations})")
    axes[1].axis("off")
    plt.show()

interact(
    show_morphology,
    operation=RadioButtons(options=["Erosion", "Dilation", "Opening", "Closing"]),
    iterations=IntSlider(min=1, max=10, value=1),
);

## Geometric Transformations

Sometimes we need to change an image's geometry rather than its pixel values: resizing, rotating, or even correcting perspective (turning an angled photo of a document into a straight-on view).

In [ ]:
from ipywidgets import FloatSlider

height, width = img.shape[:2]
center = (width // 2, height // 2)

def show_rotation(angle, scale):
    """Rotates and scales the image around its center."""
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, scale)
    rotated = cv2.warpAffine(img, rotation_matrix, (width, height))
    plt.figure(figsize=(6, 6))
    plt.imshow(rotated)
    plt.title(f"Rotated {angle} degrees, scaled x{scale}")
    plt.axis("off")
    plt.show()

interact(
    show_rotation,
    angle=IntSlider(min=-180, max=180, step=5, value=0),
    scale=FloatSlider(min=0.3, max=1.5, step=0.1, value=1.0),
);

A **perspective transform** maps four points in the original image to four new points, letting us "unwarp" an angled view. Here we manufacture an angled version of our image, then straighten it back out - the same trick used by document-scanning apps.

In [ ]:
# Manufacture a "photographed at an angle" version of the image for demonstration
src_pts = np.float32([[0, 0], [width, 0], [0, height], [width, height]])
dst_pts_skewed = np.float32([[width*0.1, height*0.05], [width*0.95, 0], [0, height], [width, height*0.9]])
skew_matrix = cv2.getPerspectiveTransform(src_pts, dst_pts_skewed)
skewed = cv2.warpPerspective(img, skew_matrix, (width, height))

# Now straighten it back out using the inverse mapping
unskew_matrix = cv2.getPerspectiveTransform(dst_pts_skewed, src_pts)
unskewed = cv2.warpPerspective(skewed, unskew_matrix, (width, height))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [img, skewed, unskewed], ["Original", "Skewed (simulated angle)", "Corrected"]):
    ax.imshow(im)
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Contours and Shape Analysis

A **contour** is a curve joining all the continuous points along a boundary with the same color or intensity - in practice, it traces the outline of shapes in a binary image. Once we have contours we can count objects, measure their area, and draw bounding boxes around them - all without any deep learning.

In [ ]:
contours, _ = cv2.findContours(otsu_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Keep only reasonably large contours (ignore tiny noise specks)
large_contours = [c for c in contours if cv2.contourArea(c) > 500]

img_with_contours = img.copy()
cv2.drawContours(img_with_contours, large_contours, -1, (255, 0, 0), 3)

print(f"Found {len(large_contours)} shapes larger than 500 pixels")

plt.figure(figsize=(8, 8))
plt.imshow(img_with_contours)
plt.title("Detected contours")
plt.axis("off")
plt.show()

**Try it yourself:** the `large_contours` list holds every shape found. For each one, use `cv2.boundingRect(contour)` (returns `x, y, w, h`) to print its bounding box and `cv2.contourArea(contour)` to print its size, largest first.

In [ ]:
for i, c in enumerate(sorted(large_contours, key=cv2.contourArea, reverse=True)):
    x, y, w, h = cv2.boundingRect(c)  # TODO: use this to print each shape's box and area
    print(f"Shape {i}: area={cv2.contourArea(c):.0f}, box=({x}, {y}, {w}, {h})")

## Feature Matching

Earlier we found *corners* - distinctive points in an image. **ORB** (Oriented FAST and Rotated BRIEF) goes a step further: it finds keypoints *and* a numeric "fingerprint" (descriptor) for each one, so we can find the *same* point again even if the image is rotated or resized. This is how panorama-stitching apps line up overlapping photos, and how old-school CV systems recognized specific objects.

In [ ]:
# Create a rotated, resized version of our image to match against
rotation_matrix = cv2.getRotationMatrix2D(center, 25, 0.8)
rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

orb = cv2.ORB_create(nfeatures=500)
kp1, des1 = orb.detectAndCompute(gray, None)
kp2, des2 = orb.detectAndCompute(cv2.cvtColor(rotated_img, cv2.COLOR_RGB2GRAY), None)

matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = sorted(matcher.match(des1, des2), key=lambda m: m.distance)

match_img = cv2.drawMatches(img, kp1, rotated_img, kp2, matches[:30], None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

print(f"Found {len(kp1)} keypoints in the original and {len(kp2)} in the rotated version")
print(f"Showing the {min(30, len(matches))} best matches")

plt.figure(figsize=(14, 7))
plt.imshow(match_img)
plt.title("ORB feature matches between original and rotated image")
plt.axis("off")
plt.show()

Notice how the matching lines correctly connect the *same* physical points on the bus even though the second image is rotated and shrunk. Stacking ideas like this - edges, corners, descriptors - is exactly how computer vision worked for decades. It took a lot of expert engineering to decide *which* features mattered for a given task. Deep learning changed that.

# Bridging to Deep Learning

A **Convolutional Neural Network (CNN)** uses the exact same convolution operation we used by hand earlier - except instead of a human choosing the kernel numbers, the network *learns* thousands of kernels automatically from data. A typical CNN layer does two things in sequence:

1. **Convolution** - slide learned kernels over the image to produce "feature maps" (just like our Sobel/blur/sharpen kernels, but learned).
2. **Pooling** - shrink each feature map by keeping only the strongest response in each small region (usually 2x2), making the network faster and a little tolerant to objects shifting slightly.

Let's build one tiny conv + pool "layer" completely by hand with NumPy, so you can see exactly what's happening under the hood before we start using industrial-strength pretrained networks.

In [ ]:
def convolve2d(image, kernel):
    """A from-scratch 2D convolution (valid padding, stride 1) for teaching purposes."""
    kh, kw = kernel.shape
    ih, iw = image.shape
    out_h, out_w = ih - kh + 1, iw - kw + 1
    output = np.zeros((out_h, out_w))
    for y in range(out_h):
        for x in range(out_w):
            region = image[y:y+kh, x:x+kw]
            output[y, x] = np.sum(region * kernel)
    return output

def max_pool2d(feature_map, size=2):
    """A from-scratch 2x2 max-pooling operation."""
    h, w = feature_map.shape
    h, w = h - h % size, w - w % size  # crop to a multiple of `size`
    cropped = feature_map[:h, :w]
    return cropped.reshape(h // size, size, w // size, size).max(axis=(1, 3))

# Work on a small crop so the pure-Python loop above finishes quickly
small_gray = cv2.resize(gray, (64, 64)).astype(np.float32) / 255.0

vertical_edge_kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
feature_map = convolve2d(small_gray, vertical_edge_kernel)
pooled = max_pool2d(feature_map, size=2)

print("Input size:", small_gray.shape)
print("After convolution:", feature_map.shape)
print("After 2x2 max pooling:", pooled.shape)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, im, title in zip(axes, [small_gray, feature_map, pooled], ["Input (64x64)", "Feature map (after conv)", "Pooled (smaller!)"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(f"{title}\n{im.shape}")
    ax.axis("off")
plt.show()

**Try it yourself:** swap `vertical_edge_kernel` for a horizontal-edge kernel (`[[-1,-1,-1],[0,0,0],[1,1,1]]`) and re-run. How does the feature map change? A real CNN has *many* kernels like this per layer, stacked into dozens of layers, all learned automatically during training - that's what we'll use for the rest of this notebook.

**What you should have seen.** The bright lines rotate. The vertical-edge kernel lights up the *upright* boundaries - the sides of the bus, the door frames, the window edges. Swap in the horizontal kernel and those go quiet, and a different set of lines appears: the roofline, the band of windows, the stripe along the side, the balconies on the building.

On this photo the horizontal kernel actually fires about **1.4x more strongly** on average than the vertical one, because a street scene full of buildings and a bus has more strong horizontal structure than vertical.

Look at the two kernels and you can see why:

```
vertical            horizontal
[-1, 0, 1]          [-1, -1, -1]
[-1, 0, 1]          [ 0,  0,  0]
[-1, 0, 1]          [ 1,  1,  1]
```

They are the same kernel, rotated. Each one subtracts one side from the other: the first compares left against right, so it only produces a big number where brightness changes *horizontally* - which is a vertical edge. The second compares top against bottom. Where the image is flat, both sides are equal, they cancel out, and you get zero - which is why smooth regions come out black.

**That is the whole idea of a CNN, and you have now built it by hand.** Nine numbers that detect one specific kind of pattern. The only thing a real network adds is that nobody chooses the nine numbers - it starts with random ones and *learns* what is worth detecting, thousands of kernels per layer, dozens of layers deep. Keep this cell in mind when the pretrained models start doing impressive things in a moment: underneath, it is this, at scale.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import urllib.request

# A few more sample images with people and everyday objects in them
urllib.request.urlretrieve("https://ultralytics.com/images/zidane.jpg", "/content/zidane.jpg")

print("Sample images ready: /content/bus.jpg, /content/zidane.jpg")

# Modern Computer Vision with YOLO11

Training a CNN from scratch takes huge datasets and a lot of compute. In practice, most people instead use **pretrained models** - networks already trained on millions of images - and simply apply them to new pictures. We'll do exactly that using [Ultralytics YOLO11](https://docs.ultralytics.com/), a fast, easy-to-use family of models that can handle several computer vision tasks with just a few lines of code.

## Image Classification

**Classification** answers one question: "what is the *main* thing in this whole image?" The model looks at the entire picture and outputs its best guess(es) at a single label, each with a confidence score.

Before we hand this over to a pretrained model, connect it back to what you just built. The convolution you wrote by hand a few cells ago - nine numbers, sliding over the image, lighting up on edges - is not an analogy for what YOLO does. It is *what YOLO does*, several million times over. YOLO11 is thousands of kernels like yours per layer, stacked dozens of layers deep, with the numbers learned from data instead of typed in by you.

That is the only difference that matters: you chose your nine numbers to find vertical edges. Nobody chose YOLO's. It worked out what was worth looking for by being shown a great many pictures.

In [ ]:
cls_model = YOLO("yolo11n-cls.pt")  # "n" = nano, the smallest and fastest variant

results = cls_model.predict(source="/content/bus.jpg")
result = results[0]

top5 = result.probs.top5
top5_conf = result.probs.top5conf

print("Top-5 predicted classes:")
for idx, conf in zip(top5, top5_conf):
    print(f"  {result.names[idx]:<20} {conf.item()*100:.1f}%")

plt.imshow(cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB))
plt.title(f"Top guess: {result.names[top5[0]]}")
plt.axis("off")
plt.show()

**Try it yourself:** run classification on `/content/zidane.jpg` as well. Does the model's top guess make sense for a photo of two people? (Classification models trained on ImageNet don't have a "person" class - see what it guesses instead, and think about *why* that might happen.)

In [ ]:
# TODO: classify zidane.jpg and print its top-5 predictions


**What you should have seen.** Something close to this:

```
suit                 34.7%
bulletproof_vest      8.6%
bow_tie               7.8%
stage                 7.0%
rugby_ball            3.6%
```

Not one of those is a person. The model looked at a photo of two footballers and confidently answered **"suit"**.

Here is why, and it is not a bug. This classifier was trained on **ImageNet-1k**, and ImageNet's 1,000 categories contain no `person`, no `man`, no `woman`, no `human` - they are genuinely not in the list. The dataset was built to tell objects and animals apart, and it is remarkably fussy about it: it has around 120 different dog breeds. People were never a category.

So the model does the only thing it can. It cannot say "person", so it names the things *around* the person that it does have words for: the **suit** they are wearing, the **bow tie**, the **stage** behind them. It is not confused and it is not failing - it is answering the only question it was ever taught to answer, about a photo nobody warned it about.

This is the single most important habit in applied machine learning: **a model can only ever answer from its own list.** It will never say "I have not been taught this". Ask an ImageNet classifier about a person and you get a confident 34.7% for "suit" - exactly the same shape of output you would get if it were right. Confidence is not knowledge, and the number tells you nothing about whether the question made sense in the first place.

That limitation is why the next sections exist. Object *detection* models are trained on COCO, which does have a `person` class - which is why YOLO finds the people in this photo when the classifier cannot.

## Object Detection

**Detection** goes further than classification: it finds *every* object of interest in the image and draws a bounding box around each one, with a class label and confidence score.

In [ ]:
det_model = YOLO("yolo11n.pt")

results = det_model.predict(source="/content/bus.jpg", conf=0.25)
result = results[0]

print(f"Found {len(result.boxes)} objects:")
for box in result.boxes:
    cls_name = result.names[int(box.cls)]
    conf = float(box.conf)
    print(f"  {cls_name:<15} confidence={conf:.2f}")

# .plot() draws the boxes and labels directly onto the image for us
annotated = result.plot()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Try lowering the confidence threshold below - notice how more (sometimes wrong) detections appear the lower you go. This is the classic **precision vs. recall** trade-off: a low threshold catches more real objects (higher recall) but also more false alarms (lower precision).

In [ ]:
from ipywidgets import FloatSlider, interact

def show_detections(confidence):
    """Re-runs detection with an adjustable confidence threshold."""
    results = det_model.predict(source="/content/bus.jpg", conf=confidence, verbose=False)
    annotated = results[0].plot()
    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"Confidence threshold = {confidence:.2f} -> {len(results[0].boxes)} objects")
    plt.axis("off")
    plt.show()

interact(show_detections, confidence=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.25));

### How do we know if a detection is "correct"? Intersection over Union (IoU)

To grade a detector, we compare its predicted box against the true ("ground truth") box using **IoU**: the area where the two boxes overlap, divided by the total area they cover together. IoU = 1.0 means a perfect match; IoU = 0 means the boxes don't overlap at all. Most benchmarks count a detection as correct if IoU > 0.5.

In [ ]:
def iou(box_a, box_b):
    """Computes Intersection over Union for two (x1, y1, x2, y2) boxes."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union_area = area_a + area_b - inter_area

    return inter_area / union_area if union_area > 0 else 0.0

# A perfect box, a slightly-off box, and a way-off box, all compared to the same ground truth
ground_truth = (100, 100, 300, 300)
close_guess = (110, 90, 310, 290)
bad_guess = (250, 250, 400, 400)

print(f"IoU (close guess):  {iou(ground_truth, close_guess):.2f}")
print(f"IoU (bad guess):    {iou(ground_truth, bad_guess):.2f}")

**Try it yourself:** grab any two boxes from the `result.boxes` object above (use `.xyxy[0].tolist()` on a box to get its `(x1, y1, x2, y2)` coordinates) and compute their IoU with the function above. Are any of the detected boxes overlapping the same object?

In [ ]:
# TODO: pick two boxes from `result.boxes` and compute their IoU


## Pose Estimation

**Pose estimation** finds specific **keypoints** on a body - shoulders, elbows, knees, and so on - and connects them into a skeleton. It's the technology behind things like sports motion analysis, fitness apps that check your squat form, and motion-capture for games and movies.

In [ ]:
pose_model = YOLO("yolo11n-pose.pt")

results = pose_model.predict(source="/content/zidane.jpg")
result = results[0]

print(f"Detected {len(result.keypoints)} people")

annotated = result.plot()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Each person's keypoints are stored as (x, y) coordinates in a fixed order (0=nose, 5=left shoulder, 7=left elbow, 9=left wrist, and so on - see the [Ultralytics pose docs](https://docs.ultralytics.com/tasks/pose/) for the full list). We can use basic trigonometry on these coordinates to measure real things, like the angle of someone's elbow.

In [ ]:
def angle_between(a, b, c):
    """Returns the angle at point b, formed by points a-b-c, in degrees."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba, bc = a - b, c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

# Keypoint indices: 5=left shoulder, 7=left elbow, 9=left wrist
keypoints = result.keypoints.xy[0].tolist()  # first detected person
shoulder, elbow, wrist = keypoints[5], keypoints[7], keypoints[9]

elbow_angle = angle_between(shoulder, elbow, wrist)
print(f"Left elbow angle: {elbow_angle:.1f} degrees")

**Try it yourself:** compute the angle at the *right* elbow instead (keypoints 6=right shoulder, 8=right elbow, 10=right wrist). Whose arm is more bent - left or right?

In [ ]:
# TODO: compute the right elbow angle using keypoints 6, 8, 10


**The answer:** the **right** arm is more bent - and the reason to look closely is what sits underneath the number.

```
LEFT  elbow (5-7-9):  171.8 deg   <- almost perfectly straight
RIGHT elbow (6-8-10): 102.9 deg   <- bent to about a right angle
```

Smaller angle means more bend, so the right arm wins comfortably. A straight arm is 180 degrees; the left one is within a few degrees of that.

**But now check the confidence, because this is the part worth taking away.** Each keypoint comes with one, and they are not the same:

```
left shoulder 0.996    right shoulder 0.955
left elbow    0.957    right elbow    0.562
left wrist    0.868    right wrist    0.502
```

The model is 96% sure where the left elbow is, and **56%** sure about the right one. The right wrist is barely better than a coin flip. That is not a flaw - the right arm is partly turned away and harder to see, and the model is telling you so.

So "102.9 degrees" is a precise-looking number built on two keypoints the model is only half confident about. It is probably about right. It is not something you would put in a physiotherapy report.

The habit: **when a model hands you coordinates, check the confidence before you do arithmetic on them.** Six significant figures of trigonometry cannot rescue an input the model was guessing at. Every number in this section had a confidence attached, and it would have been easy to never look.

## Image Segmentation

Bounding boxes are rectangles - handy, but not precise. **Segmentation** goes pixel-by-pixel: for every single pixel, the model decides which object (if any) it belongs to, producing a precise mask that traces the actual outline of each object.

In [ ]:
seg_model = YOLO("yolo11n-seg.pt")

results = seg_model.predict(source="/content/bus.jpg")
result = results[0]

print(f"Found {len(result.masks)} object masks")

annotated = result.plot()
plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### Segment Anything - Without Knowing the Classes in Advance

The segmentation model above only recognizes object classes it was trained on (people, buses, etc.). A different family of models, kicked off by Meta AI's **Segment Anything Model (SAM)**, can segment *any* object in *any* image - even ones it has never seen a label for - by using prompts like a point or a box instead of a fixed class list.

Ultralytics bundles a fast, lightweight member of that family called **FastSAM**, which we can use exactly like our other models:

In [ ]:
from ultralytics import FastSAM

fastsam_model = FastSAM("FastSAM-s.pt")

# imgsz/conf/iou are the values the Ultralytics FastSAM docs recommend. imgsz=1024 matters:
# at the default 640 the masks are noticeably coarser and split objects into more fragments.
results = fastsam_model.predict(
    source="/content/bus.jpg", device="cpu", retina_masks=True, imgsz=1024, conf=0.4, iou=0.9
)

# FastSAM is class-agnostic: it finds objects but has no class names for them, so every box
# would be labelled a meaningless "object". Turn boxes and labels off and show masks only.
annotated = results[0].plot(boxes=False, labels=False, conf=False)

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"FastSAM found {len(results[0].masks)} objects - with no class list at all")
plt.axis("off")
plt.show()


#### Prompting FastSAM

Segmenting *everything* is rarely what you want, and the picture above shows why: ~95 overlapping masks, none labelled, no way to say which one you care about.

The real power of the SAM family is that you can **prompt** it - point at something and ask "segment *that*". The [FastSAM docs](https://docs.ultralytics.com/models/fast-sam/#usage-examples) show two ways to do it. The first passes the prompt straight to `predict`:

In [ ]:
from ultralytics import FastSAM

# This is the "Predict Usage" example from the Ultralytics FastSAM docs, pointed at our image.
source = "/content/bus.jpg"

# Create a FastSAM model
model = FastSAM("FastSAM-s.pt")  # or FastSAM-x.pt

# Run inference on an image
everything_results = model(source, device="cpu", retina_masks=True, imgsz=1024, conf=0.4, iou=0.9)

# Run inference with bboxes prompt
results = model(source, bboxes=[439, 437, 524, 709])

# Run inference with points prompt
results = model(source, points=[[200, 200]], labels=[1])

print("The docs' example runs - but look at where those prompts actually land:")
print("  bboxes=[439, 437, 524, 709] ->", [int(v) for v in model(source, bboxes=[439, 437, 524, 709])[0].boxes.xyxy[0]])
print("  points=[[200, 200]]         ->", [int(v) for v in model(source, points=[[200, 200]], labels=[1])[0].boxes.xyxy[0]])


Those numbers are **placeholders**. The docs were written against an older version of `bus.jpg`, and Ultralytics has since swapped the photo - so `[439, 437, 524, 709]` no longer frames the person it used to, and `[200, 200]` lands on a balcony of the building behind the bus. The API call is right; the coordinates are meaningless for *our* image.

That is worth internalising: **a prompt only means something relative to one specific photo.** So instead of copying numbers, let's get real ones - we can let YOLO11 find an object and hand its box to FastSAM. YOLO knows *what* things are, SAM knows their exact *shape*, which is a genuinely useful pairing.

The docs' second example uses `FastSAMPredictor`, and it is the better tool here: it segments the image **once**, then lets you prompt that cached result as many times as you like instead of re-running the network per prompt.

In [ ]:
from ultralytics.models.fastsam import FastSAMPredictor
from ultralytics import YOLO

# Create FastSAMPredictor (the docs' example, with the settings from the "everything" cell above)
overrides = dict(
    conf=0.4, iou=0.9, task="segment", mode="predict", model="FastSAM-s.pt",
    save=False, imgsz=1024, retina_masks=True, device="cpu", verbose=False,
)
predictor = FastSAMPredictor(overrides=overrides)

# Segment everything - the expensive step, run ONCE
everything_results = predictor(source)
print(f"Segmented {len(everything_results[0].masks)} objects; prompting below is nearly instant.")

# Real coordinates for THIS photo, instead of the docs' placeholders
detections = YOLO("yolo11n.pt").predict(source=source, conf=0.5, verbose=False)[0]
people = [
    [int(v) for v in box.xyxy[0].tolist()]
    for box in detections.boxes
    if detections.names[int(box.cls[0])] == "person"
]
person_box = max(people, key=lambda b: (b[2] - b[0]) * (b[3] - b[1]))  # the biggest/closest person
cx, cy = (person_box[0] + person_box[2]) // 2, (person_box[1] + person_box[3]) // 2
print(f"Prompting with box {person_box} and point ({cx}, {cy})")

# Prompt inference - both reuse everything_results
bbox_results = predictor.prompt(everything_results, bboxes=[person_box])
point_results = predictor.prompt(everything_results, points=[[cx, cy]], labels=[1])


One practical snag before we look at the masks: `result.plot()` paints them **blue**, and our bus is *also* blue - so a perfectly good mask on the bus is nearly invisible. (If FastSAM has looked broken to you, this is often why.) Rather than colouring the mask, let's dim everything *outside* it, which works whatever colour the object is:

In [ ]:
def show_mask(result, ax, title, point=None, dim=0.25):
    """Show a result with everything outside its mask(s) dimmed."""
    image = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB).astype(float)
    mask = result.masks.data.any(dim=0).cpu().numpy().astype(bool)  # combine masks into one

    out = image * dim          # darken the whole image...
    out[mask] = image[mask]    # ...then restore full brightness inside the mask
    out = out.astype("uint8")

    if point:  # draw the prompt point so we can see what we asked for
        cv2.circle(out, point, 12, (255, 60, 0), -1)
        cv2.circle(out, point, 12, (255, 255, 255), 3)

    ax.imshow(out)
    ax.set_title(title)
    ax.axis("off")


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_mask(bbox_results[0], axes[0], f"Box prompt\n{len(bbox_results[0].masks)} mask(s)")
show_mask(point_results[0], axes[1], f"Point prompt at ({cx}, {cy})\n{len(point_results[0].masks)} mask(s)", point=(cx, cy))
plt.tight_layout()
plt.show()


Compare the mask counts, because the difference between the two prompts is the key idea.

The **box prompt** returns exactly **one** mask. A box says "the thing filling roughly *this* region", which is unambiguous, so FastSAM keeps the single mask that best overlaps it.

The **point prompt** returns **more than one** - and that is the honest answer, not a bug. You pointed at a person's chest: did you mean their *coat*, or the *whole person*? Both masks genuinely contain that pixel, so FastSAM gives you both. This whole-vs-part ambiguity is a well-known property of the SAM family, and it is why a box is usually the stronger prompt.

To get a single mask from a point, break the tie yourself - by area, for example:

In [ ]:
# .data is a stack of binary masks, so summing one counts its pixels = its area
areas = point_results[0].masks.data.sum(dim=(1, 2))
print("Mask areas (pixels):", sorted(int(a) for a in areas))

smallest = point_results[0][int(areas.argmin())]  # the tightest object containing the point
largest = point_results[0][int(areas.argmax())]   # the broadest object containing the point

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
show_mask(smallest, axes[0], "Smallest mask: a part (the coat)", point=(cx, cy))
show_mask(largest, axes[1], "Largest mask: the whole person", point=(cx, cy))
plt.tight_layout()
plt.show()


##### Text prompts

The docs' third prompt type describes the object in words. It is the most magical-looking and the slowest: the first call pip-installs CLIP and downloads its weights, then FastSAM scores every mask against your sentence and keeps the best match. The docs use `texts="a photo of a dog"`; there is no dog in our photo, so we ask for something that *is* here.

Because of that download this cell can take a few minutes on a CPU runtime - it is optional, so skip it if you are short on time:

In [ ]:
# Optional and slow on first run: installs CLIP, then downloads its weights (~350 MB)
text_results = predictor.prompt(everything_results, texts="a photo of a bus")

fig, ax = plt.subplots(figsize=(6, 6))
show_mask(text_results[0], ax, 'Text prompt: "a photo of a bus"')
plt.tight_layout()
plt.show()


We never told the model what a "person", a "coat" or a "bus" *is* by class id - we pointed, boxed, or described. That is the difference between this and the YOLO11 segmentation model earlier: YOLO11 only finds the 80 classes it was trained on, while FastSAM segments whatever you indicate, including things with no class name at all.

> **Try it:** move the point somewhere else (a face, a window, the pavement) and see how many masks come back. Then add a negative point - `points=[[x1, y1], [x2, y2]], labels=[1, 0]` - to say "I want this, but *not* that".
>
> **A catch worth knowing:** a box prompt picks the mask that best overlaps it, so a *huge* box works badly. Prompting with a box around the whole bus returns one small panel of it, not the bus - FastSAM has no single "whole bus" mask to match. Prompt tightly around one object.

> **Going further:** FastSAM is a great lightweight starting point. If you want to explore the cutting edge of segmentation and detection research beyond this notebook, look at:
> * [SAM 3 (Meta AI)](https://github.com/facebookresearch/sam3) - the newest, most capable Segment Anything model.
> * [RT-DETR](https://github.com/lyuwenyu/RT-DETR) - a real-time detection transformer, an alternative architecture to YOLO.
> * [Oriented Bounding Boxes (OBB)](https://docs.ultralytics.com/tasks/obb/) - a detection variant (`yolo11n-obb.pt`) that draws *rotated* boxes, useful for aerial/satellite imagery where objects appear at any angle.
>
> All three are great follow-up projects once you're comfortable with the basics above.

## Multi-Object Tracking

Detection treats every frame of a video independently - it has no memory. **Tracking** adds identity: each object gets a consistent ID number that follows it from frame to frame, which is essential for things like counting how many unique people walk past a camera, or following a specific car through traffic.

We don't have a real video handy, so let's manufacture a short one by "panning" a moving window across our bus photo - this simulates a camera moving past a scene, which is enough to demonstrate tracking IDs staying consistent across frames.

In [ ]:
# Build a synthetic panning video from our still image (no external video needed)
video_path = "/content/synthetic_pan.mp4"

# Redefine img and img_bgr to ensure they are available
img_bgr = cv2.imread("/content/bus.jpg")
img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

height, width = img.shape[:2]
crop_w, crop_h = width // 2, height
num_frames = 20
max_shift = width - crop_w

writer = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*"mp4v"), 5, (crop_w, crop_h))
for i in range(num_frames):
    x_offset = int(max_shift * i / (num_frames - 1))
    frame = img_bgr[:, x_offset:x_offset + crop_w]
    writer.write(frame)
writer.release()

print(f"Wrote a {num_frames}-frame synthetic panning video to {video_path}")

In [ ]:
track_results = list(det_model.track(source=video_path, tracker="bytetrack.yaml", persist=True, verbose=False, stream=True))

# Show every 4th frame so we can see the same track ID follow the bus across the pan
sample_indices = range(0, len(track_results), 4)
fig, axes = plt.subplots(1, len(list(sample_indices)), figsize=(16, 4))
for ax, i in zip(axes, sample_indices):
    annotated = track_results[i].plot()
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {i}")
    ax.axis("off")
plt.show()

Notice the small ID number next to each box - it should stay the *same* for the same object across frames, even though its position and how much of it is visible changes as the camera pans.

# The Hugging Face Model Hub

So far we've used one model family (Ultralytics YOLO11) for everything. In the real world, you'll want to pull in models built by other researchers and companies too. [**Hugging Face**](https://huggingface.co/models) hosts the largest public hub of pretrained AI models - well over a million, covering vision, language, and audio - all downloadable and usable with a few lines of code via the `transformers` library.

Open [huggingface.co/models](https://huggingface.co/models) in a new tab and look at the left sidebar: you can filter by task (**Image Classification**, **Object Detection**, **Image Segmentation**, and many more), and each model has a page showing how many times it's been downloaded, a description, and a "model card" explaining what it was trained on and its limitations. This is worth exploring before you pick a model for a real project.

In [ ]:
!pip install transformers

## Image Classification with a Hugging Face Model

The `pipeline` function is the fastest way to use any model from the Hub - give it a task name and (optionally) a specific model ID copied straight from the model's page, and it handles downloading and running the model for you. Let's try [`google/vit-base-patch16-224`](https://huggingface.co/google/vit-base-patch16-224), a Vision Transformer (ViT) - a different architecture from the CNNs we've used so far, but solving the same classification task.

In [ ]:
from transformers import pipeline
from PIL import Image

hf_classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

pil_image = Image.open("/content/bus.jpg")
predictions = hf_classifier(pil_image)

for pred in predictions:
    print(f"  {pred['label']:<30} {pred['score']*100:.1f}%")

## Zero-Shot Classification with CLIP

Every classifier we've used so far can only recognize the fixed list of classes it was trained on. [**CLIP**](https://huggingface.co/openai/clip-vit-base-patch32) (from OpenAI, hosted on Hugging Face) was trained differently - on image/caption pairs from across the internet - so it can classify images against *any* list of labels you type in yourself, with no retraining at all. This is called **zero-shot** classification.

In [ ]:
zero_shot_classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

candidate_labels = ["a bus", "a bicycle", "a cat", "a busy city street"]
predictions = zero_shot_classifier(pil_image, candidate_labels=candidate_labels)

for pred in predictions:
    print(f"  {pred['label']:<25} {pred['score']*100:.1f}%")

**Try it yourself:** change `candidate_labels` to a list of your own made-up categories (nothing CLIP was ever explicitly "trained" to recognize as a class, like `"a peaceful morning"` or `"a scene from a movie"`) and see how it scores them against `zidane.jpg`.

In [ ]:
# TODO: write your own candidate_labels list and run zero-shot classification on zidane.jpg


## Object Detection with a Hugging Face Model

Just like classification, object detection has a `pipeline` too. Let's compare [`facebook/detr-resnet-50`](https://huggingface.co/facebook/detr-resnet-50) (DEtection TRansformer, a very different architecture from YOLO) against the YOLO11 detections from the object detection section earlier.

In [ ]:
from PIL import ImageDraw

hf_detector = pipeline("object-detection", model="facebook/detr-resnet-50")
detections = hf_detector(pil_image)

print(f"DETR found {len(detections)} objects:")
annotated_pil = pil_image.copy()
draw = ImageDraw.Draw(annotated_pil)
for det in detections:
    print(f"  {det['label']:<15} confidence={det['score']:.2f}")
    box = det["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=3)
    draw.text((box["xmin"], box["ymin"] - 10), det["label"], fill="red")

plt.figure(figsize=(8, 8))
plt.imshow(annotated_pil)
plt.axis("off")
plt.show()

**Try it yourself:** head to [huggingface.co/models](https://huggingface.co/models?pipeline_tag=object-detection), filter by the **Object Detection** task, pick a different model, and swap its model ID into the `pipeline(...)` call above. Does it find the same objects as DETR and YOLO11?

# Fine-Tuning a Model (Transfer Learning)

So far we've only used models exactly as they came, pretrained on huge public datasets. **Fine-tuning** (a.k.a. transfer learning) takes a pretrained model and continues training it for a few epochs on your *own*, much smaller dataset - the model already knows general shapes and textures, so it only needs a little extra training to adapt to a new task.

To keep this quick enough to run live in class on a CPU, we'll fine-tune a classifier on **MNIST160** - a tiny 160-image sample of handwritten digits that Ultralytics provides specifically for fast demos like this.

In [ ]:
finetune_model = YOLO("yolo11n-cls.pt")

# A handful of epochs on a tiny dataset - just to see the training loop in action, not for accuracy
finetune_results = finetune_model.train(data="mnist160", epochs=5, imgsz=32, verbose=False)

In [ ]:
# mnist160 ships with train/ and test/ only - no val split - so ask for test explicitly
metrics = finetune_model.val(split="test")
print("Top-1 accuracy on the test set:", metrics.top1)
print("Top-5 accuracy on the test set:", metrics.top5)

**Try it yourself:** re-run the training cell with `epochs=15` instead of `5` (and maybe `imgsz=64`). Does test accuracy improve? At what point do you think it would start to plateau?

**What you should have seen.** Accuracy goes up when you go from 5 epochs to 15, and then the gains get smaller and smaller. The exact numbers depend on the random seed, the split, and the GPU you happened to get, so yours will not match anyone else's - but the *shape* is always the same, and the shape is the lesson.

Why it plateaus: an epoch is one pass over the training data. The first few passes are where the model picks up the obvious, general stuff - the edges, colours and shapes that genuinely separate your classes. After that there is nothing general left to learn, and the only way to keep improving on the *training* set is to start memorising individual training images.

That is the moment to watch for, because training accuracy will keep climbing happily while test accuracy flattens or gets *worse*. The gap between those two curves is overfitting, and it is what the train/test split exists to expose. More epochs is not more learning - past a point it is just more memorising.

# Real-World Mini Projects

Let's combine what we've learned into a few small, practical applications.

## Webcam Capture (Colab only)

Colab notebooks can access your webcam directly through the browser using a bit of JavaScript. Run the cell below, grant camera permission, and click **Capture** to take a photo - then we'll run our detection model on it. (Skip this section if you don't have a webcam available - everything else works fine on `bus.jpg`/`zidane.jpg`.)

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="/content/webcam.jpg", quality=0.8):
    """Captures a single photo from the browser's webcam (Colab only)."""
    js = Javascript('''
        async function takePhoto(quality) {
          const div = document.createElement('div');
          const capture = document.createElement('button');
          capture.textContent = 'Capture';
          div.appendChild(capture);

          const video = document.createElement('video');
          video.style.display = 'block';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});

          document.body.appendChild(div);
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();

          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
          await new Promise((resolve) => capture.onclick = resolve);

          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          stream.getVideoTracks()[0].stop();
          div.remove();
          return canvas.toDataURL('image/jpeg', quality);
        }
        ''')
    display(js)
    data = eval_js(f"takePhoto({quality})")
    binary = b64decode(data.split(",")[1])
    with open(filename, "wb") as f:
        f.write(binary)
    return filename

# Uncomment to try it (needs a webcam and browser permission):
# photo_path = take_photo()
# plt.imshow(cv2.cvtColor(cv2.imread(photo_path), cv2.COLOR_BGR2RGB))
# plt.axis("off")
# plt.show()

## Mini Project: Object Counter

A very common real-world task: "how many of *this specific thing* are in the image?" (cars in a parking lot, people in a queue, defective parts on a conveyor belt). We already have everything we need from the object detection section earlier - we just need to filter and count.

In [ ]:
from ipywidgets import Dropdown

available_classes = sorted(set(det_model.names.values()))

def count_objects(class_name, confidence):
    """Counts how many objects of a chosen class are detected above a confidence threshold."""
    results = det_model.predict(source="/content/bus.jpg", conf=confidence, verbose=False)
    boxes = results[0].boxes
    count = sum(1 for b in boxes if det_model.names[int(b.cls)] == class_name)

    annotated = results[0].plot()
    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"{class_name}: {count} found (confidence > {confidence:.2f})")
    plt.axis("off")
    plt.show()

interact(
    count_objects,
    class_name=Dropdown(options=available_classes, value="person", description="Class:"),
    confidence=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.25),
);

## Mini Project: Privacy Blur

Photos posted publicly often need to have faces or people blurred out for privacy. Instead of a dedicated face detector, we can reuse our object detector: find every `person` box, and blur just that region of the image.

In [ ]:
def blur_people(image_path, confidence=0.25, blur_strength=51):
    """Detects people and blurs just their bounding-box regions for privacy."""
    image_bgr = cv2.imread(image_path)
    results = det_model.predict(source=image_path, conf=confidence, classes=[0], verbose=False)  # class 0 = person

    blurred = image_bgr.copy()
    k = blur_strength if blur_strength % 2 == 1 else blur_strength + 1
    for box in results[0].boxes.xyxy:
        x1, y1, x2, y2 = map(int, box)
        region = blurred[y1:y2, x1:x2]
        blurred[y1:y2, x1:x2] = cv2.GaussianBlur(region, (k, k), 0)

    return cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
original_zidane = cv2.cvtColor(cv2.imread("/content/zidane.jpg"), cv2.COLOR_BGR2RGB)
axes[0].imshow(original_zidane)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(blur_people("/content/zidane.jpg"))
axes[1].set_title("People blurred")
axes[1].axis("off")
plt.show()

## Mini Project: Instagram-Style Photo Filters

Finally, let's combine several of the classical image-processing techniques from earlier into a single fun, adjustable "filter picker" - the same basic idea behind photo-editing app filters.

In [ ]:
def apply_filter(filter_name):
    """Applies a chosen classic image-processing filter to the sample image."""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    sharpen_kernel = np.array([
        [ 0, -1,  0],
        [-1,  5, -1],
        [ 0, -1,  0],
    ])
    if filter_name == "Original":
        result = img
    elif filter_name == "Grayscale":
        result = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    elif filter_name == "Sepia":
        sepia_matrix = np.array([[0.393, 0.769, 0.189],
                                  [0.349, 0.686, 0.168],
                                  [0.272, 0.534, 0.131]])
        result = np.clip(img @ sepia_matrix.T, 0, 255).astype(np.uint8)
    elif filter_name == "Soft Blur":
        result = cv2.GaussianBlur(img, (15, 15), 0)
    elif filter_name == "Sketch (edges)":
        edges = cv2.Canny(gray, 60, 150)
        result = cv2.cvtColor(255 - edges, cv2.COLOR_GRAY2RGB)
    elif filter_name == "Vivid (sharpen)":
        result = cv2.filter2D(img, -1, sharpen_kernel)

    plt.figure(figsize=(6, 6))
    plt.imshow(result)
    plt.title(filter_name)
    plt.axis("off")
    plt.show()

interact(
    apply_filter,
    filter_name=Dropdown(
    options=["Original", "Grayscale", "Sepia", "Soft Blur", "Sketch (edges)", "Vivid (sharpen)"],
    description="Filter:",
));

**Try it yourself:** add one more option to `apply_filter` - for example a "Cool Tone" filter that boosts the blue channel and reduces the red channel. Add it to both the `if/elif` chain and the dropdown `options` list.

#  Generative Vision
Everything so far has been about *understanding* an existing image. Generative models flip that around - they *create* a brand new image from a text description. This section is optional and needs a GPU runtime (**Runtime > Change runtime type > T4 GPU** in Colab) - feel free to skip it if you're low on GPU quota.

In [ ]:
# Uncomment the line below the first time you run this section
!pip install diffusers transformers accelerate torch torchvision

import torch
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

prompt = "a friendly robot painting a watercolor landscape, studio lighting"
image = pipe(prompt, num_inference_steps=25).images[0]
image

# Homework

Upload a photo of your own to `/content/` (use the folder icon on the left of Colab) and use it in place of `bus.jpg` / `zidane.jpg` wherever it makes sense.

**Warm-up, classical CV:**
1. **Color hunter**: use the HSV color-masking widget from the color spaces section to isolate a specific colored object in a photo of your own.
2. **Edge artist**: use the Canny widget from the edge detection section to find threshold values that cleanly trace the outline of one object, then apply the "Sketch" filter from the photo-filters mini project to the same photo and compare.
3. **Shape counter**: use contours from the shape-analysis section to count how many distinct shapes/objects appear in a simple photo (e.g. a handful of coins or buttons on a plain background).

**Core, deep learning tasks:**
4. **Try them all**: run classification, detection, pose estimation, and segmentation on a photo of your own. Which task gave the most useful result for *your* photo, and why?
5. **Bigger is... slower?**: re-run object detection using `"yolo11s.pt"` instead of `"yolo11n.pt"`. Compare the results and how long each takes to run. Is the trade-off worth it?
6. **Build your own mini project**: combine the object counter and privacy blur ideas from the mini projects section into one function that blurs everything *except* a chosen class (e.g. blur everyone except the tallest person).

**Stretch, go further:**
7. **Hub explorer**: browse [huggingface.co/models](https://huggingface.co/models), filter by **Image Segmentation**, pick a model you haven't used yet, and run it on `bus.jpg` via `pipeline(...)`. How does it compare to the YOLO11 and FastSAM segmentation results from earlier?
8. **Prompted segmentation**: look up how to give FastSAM a `points` or `bboxes` prompt (see the [FastSAM docs](https://docs.ultralytics.com/models/fast-sam/)) and segment just *one* chosen object instead of everything in the image.
9. **Longer fine-tune**: repeat the fine-tuning exercise with more epochs and a bigger `imgsz`, and plot how validation accuracy changes as training progresses.
10. **OBB explorer**: read the [Oriented Bounding Box docs](https://docs.ultralytics.com/tasks/obb/) and explain, in your own words (a markdown cell is fine), why aerial/satellite imagery needs rotated boxes instead of the axis-aligned boxes we used in the object detection section.

In [ ]:
# Warm-up Challenge 1: Color hunter
# Upload your own photo to /content/ first, then point my_photo at it.
my_photo = "/content/bus.jpg"  # TODO: replace with your own uploaded photo

img_bgr_c1 = cv2.imread(my_photo)
img_hsv_c1 = cv2.cvtColor(img_bgr_c1, cv2.COLOR_BGR2HSV)

# TODO: pick HSV lower/upper bounds for the color you want to isolate
# (reuse the color_mask widget above to find good values first)
lower_c1 = np.array([0, 0, 0])
upper_c1 = np.array([179, 255, 255])

mask_c1 = cv2.inRange(img_hsv_c1, lower_c1, upper_c1)
result_c1 = cv2.bitwise_and(img_bgr_c1, img_bgr_c1, mask=mask_c1)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(result_c1, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Isolated color")
plt.show()


In [ ]:
# Warm-up Challenge 2: Edge artist
my_photo = "/content/bus.jpg"  # TODO: replace with your own uploaded photo

img_bgr_c2 = cv2.imread(my_photo)
gray_c2 = cv2.cvtColor(img_bgr_c2, cv2.COLOR_BGR2GRAY)

# TODO: tune these thresholds (use the Canny widget above to explore first)
low_threshold_c2 = 50
high_threshold_c2 = 150
edges_c2 = cv2.Canny(gray_c2, low_threshold_c2, high_threshold_c2)

# TODO: apply the "Sketch" filter from the mini project section to the same photo
# sketch_c2 = apply_filter("Sketch")  # only works if my_photo == img from earlier

plt.figure(figsize=(6, 6))
plt.imshow(edges_c2, cmap="gray")
plt.axis("off")
plt.title("Canny edges")
plt.show()


In [ ]:
# Warm-up Challenge 3: Shape counter
my_photo = "/content/bus.jpg"  # TODO: replace with a photo of coins/buttons/simple shapes

img_bgr_c3 = cv2.imread(my_photo)
gray_c3 = cv2.cvtColor(img_bgr_c3, cv2.COLOR_BGR2GRAY)

# Otsu's method picks a threshold automatically
_, binary_c3 = cv2.threshold(gray_c3, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
contours_c3, _ = cv2.findContours(binary_c3, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# TODO: tune this to ignore tiny noise specks
min_area_c3 = 200
shapes_c3 = [c for c in contours_c3 if cv2.contourArea(c) > min_area_c3]

print(f"Found {len(shapes_c3)} shapes")


In [ ]:
# Core Challenge 4: Try them all
my_photo = "/content/bus.jpg"  # TODO: replace with your own uploaded photo

cls_model_c4 = YOLO("yolo11n-cls.pt")
det_model_c4 = YOLO("yolo11n.pt")
pose_model_c4 = YOLO("yolo11n-pose.pt")
seg_model_c4 = YOLO("yolo11n-seg.pt")

# TODO: run predict() with each model on my_photo and .plot() the results
# classification_result = cls_model_c4.predict(source=my_photo)[0]
# detection_result = det_model_c4.predict(source=my_photo)[0]
# pose_result = pose_model_c4.predict(source=my_photo)[0]
# segmentation_result = seg_model_c4.predict(source=my_photo)[0]

# TODO: which task gave the most useful result for your photo, and why?


In [ ]:
# Core Challenge 5: Bigger is... slower?
import time

my_photo = "/content/bus.jpg"  # TODO: replace with your own uploaded photo

small_model_c5 = YOLO("yolo11n.pt")
big_model_c5 = YOLO("yolo11s.pt")

start_c5 = time.time()
small_results_c5 = small_model_c5.predict(source=my_photo, conf=0.25)
small_time_c5 = time.time() - start_c5

start_c5 = time.time()
big_results_c5 = big_model_c5.predict(source=my_photo, conf=0.25)
big_time_c5 = time.time() - start_c5

print(f"nano:  {len(small_results_c5[0].boxes)} objects in {small_time_c5:.2f}s")
print(f"small: {len(big_results_c5[0].boxes)} objects in {big_time_c5:.2f}s")

# TODO: is the extra accuracy worth the extra time, in your opinion?


In [ ]:
# Core Challenge 6: Build your own mini project
my_photo = "/content/bus.jpg"  # TODO: replace with your own uploaded photo

def blur_all_except(image_path, keep_class):
    """Blurs every detected object except the chosen class."""
    model = YOLO("yolo11n.pt")
    image = cv2.imread(image_path)
    results = model.predict(source=image_path, conf=0.25)[0]

    for box in results.boxes:
        class_name = model.names[int(box.cls[0])]
        # TODO: skip blurring when class_name == keep_class
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        region = image[y1:y2, x1:x2]
        image[y1:y2, x1:x2] = cv2.GaussianBlur(region, (51, 51), 0)

    return image

# TODO: call blur_all_except(my_photo, "person") and display the result


In [ ]:
# Stretch Challenge 7: Hub explorer
from transformers import pipeline
from PIL import Image

# TODO: browse huggingface.co/models, filter by "Image Segmentation",
# and paste a model id you haven't used yet below
hf_model_id_c7 = "REPLACE_ME"

segmenter_c7 = pipeline("image-segmentation", model=hf_model_id_c7)
pil_image_c7 = Image.open("/content/bus.jpg")
segments_c7 = segmenter_c7(pil_image_c7)

print(f"Found {len(segments_c7)} segments")

# TODO: how does this compare to the YOLO11 and FastSAM segmentation results?


In [ ]:
# Stretch Challenge 8: Prompted segmentation
from ultralytics import FastSAM

fastsam_model_c8 = FastSAM("FastSAM-s.pt")

# TODO: pass a points= / bboxes= / texts= prompt to segment just one chosen
# object instead of everything (see the prompting section earlier in this notebook)
results_c8 = fastsam_model_c8.predict(source="/content/bus.jpg")

annotated_c8 = results_c8[0].plot(labels=False, conf=False)
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(annotated_c8, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
# Stretch Challenge 9: Longer fine-tune
finetune_model_c9 = YOLO("yolo11n-cls.pt")

# TODO: increase epochs and imgsz compared to the earlier fine-tuning demo
finetune_results_c9 = finetune_model_c9.train(data="mnist160", epochs=5, imgsz=32)

# TODO: plot how validation accuracy (top1/top5) changes across epochs
# hint: finetune_results_c9 has a save_dir with a results.csv you can load with pandas


# Stretch Challenge 10 - write your answer here as markdown

### References

- [Ultralytics YOLO11 Documentation](https://docs.ultralytics.com/)
- [Ultralytics Tasks Overview](https://docs.ultralytics.com/tasks/)
- [Ultralytics Tracking Mode](https://docs.ultralytics.com/modes/track/)
- [FastSAM](https://docs.ultralytics.com/models/fast-sam/)
- [Hugging Face Model Hub](https://huggingface.co/models)
- [Hugging Face Transformers `pipeline` Documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)
- [Segment Anything Model 3 (Meta AI)](https://github.com/facebookresearch/sam3)
- [RT-DETR](https://github.com/lyuwenyu/RT-DETR)
- [OpenCV Documentation](https://docs.opencv.org/)